# Text Summarization with Transformers — fine-tuning BART on CNN/DailyMail

**Deep Learning course project** · Kumar Sidharth · B.Tech CSE, BIT Mesra · [github.com/ksidharth8/text-summarization-transformers](https://github.com/ksidharth8/text-summarization-transformers)

Self-contained training + evaluation pipeline. It compares **extractive** summarizers (LEAD-3, TextRank, transformer-embedding MMR) with an **abstractive** transformer (`facebook/bart-base`, fine-tuned here).

| § | Step | Main output |
|---|---|---|
| 1 | Setup, configuration, project modules (`src/`) | |
| 2 | Load + clean the dataset, statistics | `results/figures/fig_lengths.png` |
| 3 | Extractive baselines on the test set | ROUGE table |
| 4 | Fine-tune BART-base (fp16, time-budgeted LR schedule) | `results/figures/fig_training_curves.png` |
| 5 | Save the model (Hugging Face format + zip) | `bart-base-cnn-summarizer/`, `bart-base-cnn-summarizer.zip` |
| 6 | Evaluation: ROUGE-1/2/L/Lsum + 95% CIs, zero-shot vs fine-tuned, significance, abstractiveness | `results/metrics.json`, figures |
| 7 | Qualitative examples + custom test cases (incl. long-document mode) | `results/custom_cases.md` |

### How to run on Kaggle
1. **Add Input** → search `newspaper-text-summarization-cnn-dailymail` (by *gowrishankarp*) → **Add**.
2. **Settings** → *Accelerator*: **GPU T4 x2** · *Internet*: **On** (a phone-verified account is required; the internet is used to download `facebook/bart-base`).
3. Leave `RUN_MODE = "smoke"` and click **Run All** (≈10–15 min). This tests data, GPU, internet and every step on small subsets.
4. Change to `RUN_MODE = "full"`, then **Save Version → Save & Run All (Commit)**. Runs in the background (≈5.5–6.5 h; training itself is capped by `train_hours`). You can close the browser.
5. When the version is complete: **Output** tab → download `bart-base-cnn-summarizer.zip` and `results/`.

In [ ]:
# 1. Setup --------------------------------------------------------------------------------------
# Kaggle images already contain torch / transformers / datasets / accelerate; only small extras may be missing.
import importlib.util, subprocess, sys
for pip_name, module in [("rouge-score", "rouge_score"), ("nltk", "nltk")]:
    if importlib.util.find_spec(module) is None:
        print(f"installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

In [ ]:
import os, json

# "smoke": ~10-15 min end-to-end check on small subsets (the scores are NOT meaningful)
# "full" : real training on all ~287k training articles + evaluation on the full 11,490-article test set
RUN_MODE = os.environ.get("SUMM_RUN_MODE", "smoke")

CFG = dict(
    model_name="facebook/bart-base",       # alternatives: "facebook/bart-large" (per_device_batch=4), "t5-small", "google/flan-t5-base"
    embed_model="sentence-transformers/all-MiniLM-L6-v2",   # sentence encoder for the embedding-MMR baseline
    seed=42,
    # data
    train_samples=None, val_samples=2000, test_samples=None,  # None = the full split
    max_source_len=512, max_target_len=128,
    # optimisation: training stops after `epochs` OR `train_hours`, whichever comes first (the LR schedule adapts)
    epochs=3, train_hours=4.5,
    lr=5e-5, weight_decay=0.01, warmup_steps=500, label_smoothing=0.1, max_grad_norm=1.0,
    per_device_batch=16, effective_batch=32, fp16=True,       # CUDA out of memory? -> per_device_batch=8
    eval_every=1000, log_every=100,
    # decoding (CNN/DailyMail settings of the BART paper)
    gen_batch=32, num_beams=4, max_new_tokens=142, min_new_tokens=56, length_penalty=2.0, no_repeat_ngram_size=3,
    # evaluation extras
    zero_shot_samples=500, novelty_samples=2000, run_embed_extractive=True, n_examples=5,
    push_to_hub=False, hub_repo="ksidharth8/bart-base-cnn-dailymail",   # needs a Kaggle secret named HF_TOKEN
)
SMOKE = dict(train_samples=2000, val_samples=200, test_samples=200, epochs=1, train_hours=0.12, warmup_steps=20,
             eval_every=40, log_every=10, zero_shot_samples=40, novelty_samples=200)
if RUN_MODE == "smoke":
    CFG.update(SMOKE)
# environment overrides (used to test this notebook offline with tiny models)
if os.environ.get("SUMM_MODEL_NAME"):
    CFG["model_name"] = os.environ["SUMM_MODEL_NAME"]
if os.environ.get("SUMM_EMBED_MODEL"):
    CFG["embed_model"] = os.environ["SUMM_EMBED_MODEL"]
CFG.update(json.loads(os.environ.get("SUMM_CFG_OVERRIDES", "{}")))
print("RUN_MODE =", RUN_MODE)
print(json.dumps(CFG, indent=1))

In [ ]:
import os, sys, re, gc, glob, json, math, time, shutil, inspect, zipfile, platform, dataclasses, warnings, datetime
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

transformers.logging.set_verbosity_error()
try:
    datasets.utils.logging.set_verbosity_error()
    datasets.disable_progress_bars()
except Exception:
    pass

IN_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = os.path.abspath(os.environ.get("SUMM_WORK_DIR", "/kaggle/working" if IN_KAGGLE else "outputs"))
DATA_DIR = os.environ.get("SUMM_DATA_DIR", "/kaggle/input" if IN_KAGGLE else "data")
CKPT_DIR = os.environ.get("SUMM_CKPT_DIR", "/tmp/checkpoints")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
for d in (WORK_DIR, os.path.join(WORK_DIR, "src"), os.path.join(WORK_DIR, "test_cases"), RESULTS_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
MODEL_TAG = os.path.basename(CFG["model_name"].rstrip("/"))
MODEL_DIR = os.path.join(WORK_DIR, f"{MODEL_TAG}-cnn-summarizer")
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

N_GPU = torch.cuda.device_count()
GPU_NAME = torch.cuda.get_device_name(0) if N_GPU else "none"
if RUN_MODE == "full" and N_GPU == 0:
    raise RuntimeError("No GPU visible. Kaggle: Settings -> Accelerator -> GPU T4 x2 (then run again).")
transformers.set_seed(CFG["seed"])
NUM_PROC = max(1, min(4, os.cpu_count() or 1))
NUM_WORKERS = min(2, max(0, (os.cpu_count() or 1) - 1))
ENV = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
           datasets=datasets.__version__, numpy=np.__version__, pandas=pd.__version__, cuda=torch.version.cuda,
           gpu=f"{GPU_NAME} x{N_GPU}" if N_GPU else "CPU only", cpus=os.cpu_count())
FIGURES = {}
T_NOTEBOOK = time.time()
print(json.dumps(ENV, indent=1))
print("WORK_DIR:", WORK_DIR, "| DATA_DIR:", DATA_DIR)

### Project modules
The cells below write the project's Python package to `src/` (identical to the GitHub repository), so the notebook output is a complete, importable package next to the trained model.

In [ ]:
%%writefile {WORK_DIR}/src/__init__.py
"""Text summarization toolkit (Deep Learning course project).

Modules
-------
preprocessing   cleaning of CNN/DailyMail boilerplate, rule-based sentence splitting
extractive      LEAD-k, TextRank and transformer-embedding (centroid + MMR) extractive summarizers
metrics         ROUGE with bootstrap confidence intervals, novel n-grams, length statistics
summarizer      inference wrapper for fine-tuned BART/T5/PEGASUS models, incl. long-document mode
"""

__version__ = "1.0.0"

In [ ]:
%%writefile {WORK_DIR}/src/preprocessing.py
"""Text cleaning and sentence splitting (pure Python, no downloads needed).

CNN/DailyMail articles carry scraping boilerplate that hurts extractive baselines and wastes
encoder tokens, e.g.

    "By . Associated Press . PUBLISHED: . 14:11 EST, 25 October 2013 . | . UPDATED: . ..."
    "WASHINGTON (CNN) -- ..."

`clean_article` removes these conservatively (only well-defined patterns). Reference summaries
are never changed in content: `clean_summary` only normalises whitespace and keeps one
highlight per line, which is what ROUGE-Lsum expects.
"""
from __future__ import annotations

import re

__all__ = ["clean_article", "clean_summary", "split_sentences", "word_tokenize", "count_words"]

# DailyMail time stamps: "PUBLISHED: . 14:11 EST, 25 October 2013 . | . "
_TIMESTAMP = re.compile(
    r"(?:PUBLISHED|UPDATED)\s*:\s*\.?\s*\d{1,2}:\d{2}\s*(?:[AaPp]\.?[Mm]\.?\s*)?(?:[A-Z]{2,4}\s*)?,?\s*"
    r"\d{1,2}\s+[A-Za-z]+\s+\d{4}\s*\.?\s*(?:\|\s*\.?\s*)?"
)
# "Last updated at 3:23 PM on 12th January 2012 ."
_LAST_UPDATED = re.compile(
    r"Last updated (?:at|on)\s*\d{1,2}:\d{2}\s*(?:[AaPp]\.?[Mm]\.?)?\s*(?:on\s*)?"
    r"\d{1,2}(?:st|nd|rd|th)?\s+[A-Za-z]+\s+\d{4}\s*\.?\s*"
)
# "By . Daily Mail Reporter . " / "By . Jane Doe . and John Roe . " (only at the very start)
_BYLINE = re.compile(r"^By\s*\.\s*[^.]{1,100}?\s*\.\s*(?:(?:and|&)\s*[^.]{1,100}?\s*\.\s*)?")
# "(CNN) -- ", "LONDON, England (CNN) -- ", "(CNN)Hillary ..." (only at the very start)
_DATELINE = re.compile(   # place name = up to 6 capitalised words ("LONDON, England", "Hong Kong")
    r"^(?:[A-Z][A-Za-z.'-]*(?:,?\s+[A-Z][A-Za-z.'-]*){0,5},?\s*)?"
    r"\((?:CNN|CNN Student News|EW\.com|Mashable|Reuters|AP)\)\s*(?:--|-|\u2014|\u2013)?\s*"
)
_SCROLL = re.compile(r"\bScroll down for (?:video|videos)\s*\.?\s*", re.IGNORECASE)
_SPACE_BEFORE_PUNCT = re.compile(r"\s+([.!?,;:])(?=\s|$)")
_WS = re.compile(r"[ \t\r\f\v]+")
_ANY_WS = re.compile(r"\s+")

# Tokens that end with '.' but usually do not end a sentence.
_ABBREVIATIONS = frozenset(
    """mr mrs ms dr prof sr jr st mt vs etc inc ltd co corp jan feb mar apr jun jul aug sep sept oct
    nov dec gen gov sen rep col lt sgt capt cmdr adm rev fig figs eq eqs al approx dept univ ave blvd
    no nos vol pp ch sec u.s u.k u.n e.g i.e a.m p.m ph.d""".split()
)
_BOUNDARY = re.compile(r"[.!?]+[\"'\u201d\u2019)\]]*(?=\s+[\"'\u201c\u2018(\[]?[A-Z0-9])")
_WORD = re.compile(r"[a-z0-9]+")


def clean_article(text: str) -> str:
    """Remove CNN/DailyMail scraping boilerplate and normalise whitespace (single line output)."""
    if not isinstance(text, str):
        return ""
    t = _ANY_WS.sub(" ", text).strip()
    head, tail = t[:600], t[600:]            # boilerplate only ever appears at the top
    head = _TIMESTAMP.sub("", head)
    head = _LAST_UPDATED.sub("", head)
    head = _BYLINE.sub("", head.lstrip())
    head = _DATELINE.sub("", head.lstrip())
    t = head + tail
    t = _SCROLL.sub("", t)
    t = _SPACE_BEFORE_PUNCT.sub(r"\1", t)
    return _ANY_WS.sub(" ", t).strip()


def clean_summary(text: str) -> str:
    """Normalise a reference summary: one sentence/highlight per line, tidy spacing."""
    if not isinstance(text, str):
        return ""
    lines = []
    for line in text.replace("\r", "\n").split("\n"):
        line = _WS.sub(" ", line).strip()
        line = _SPACE_BEFORE_PUNCT.sub(r"\1", line)
        if line:
            lines.append(line)
    return "\n".join(lines)


def split_sentences(text: str) -> list[str]:
    """Abbreviation-aware rule-based sentence splitter. Newlines are always sentence boundaries."""
    if not isinstance(text, str) or not text.strip():
        return []
    sentences: list[str] = []
    for line in re.split(r"\s*\n+\s*", text.strip()):
        line = _ANY_WS.sub(" ", line).strip()
        if not line:
            continue
        start = 0
        for m in _BOUNDARY.finditer(line):
            prev_token = line[start:m.start()].rsplit(" ", 1)[-1].lower().strip("\"'(\u201c\u2018")
            if (
                prev_token in _ABBREVIATIONS
                or re.fullmatch(r"[a-z]", prev_token)                 # initials: "J. K. Rowling"
                or re.fullmatch(r"(?:[a-z]\.)+[a-z]", prev_token)     # "U.S", "e.g"
            ):
                continue
            sent = line[start:m.end()].strip()
            if sent:
                sentences.append(sent)
            start = m.end()
        tail = line[start:].strip()
        if tail:
            sentences.append(tail)
    return sentences


def word_tokenize(text: str) -> list[str]:
    """Lower-cased alphanumeric tokens (same normalisation as the rouge-score tokenizer)."""
    return _WORD.findall(text.lower()) if isinstance(text, str) else []


def count_words(text: str) -> int:
    return len(text.split()) if isinstance(text, str) else 0

In [ ]:
%%writefile {WORK_DIR}/src/extractive.py
"""Extractive summarization methods.

* LEAD-k          first k sentences (exploits the "inverted pyramid" style of news writing)
* TextRank        PageRank over a sentence-similarity graph (Mihalcea & Tarau, 2004)
* Embedding MMR   transformer sentence embeddings (MiniLM) -> centroid relevance + Maximal
                  Marginal Relevance for diversity (Carbonell & Goldstein, 1998)

All methods return the selected sentences in their original document order, one per line.
"""
from __future__ import annotations

import numpy as np

from .preprocessing import split_sentences, word_tokenize

__all__ = ["lead_k", "textrank", "textrank_scores", "mmr_select", "EmbeddingExtractor", "STOPWORDS"]

STOPWORDS = frozenset(
    """a about above after again against all am an and any are as at be because been before being below
    between both but by can could did do does doing down during each few for from further had has have
    having he her here hers herself him himself his how i if in into is it its itself just me more most my
    myself no nor not now of off on once only or other our ours ourselves out over own same she should so
    some such than that the their theirs them themselves then there these they this those through to too
    under until up very was we were what when where which while who whom why will with would you your
    yours yourself yourselves s t said says also one two new year years mr mrs ms""".split()
)


def _candidate_indices(sentences: list[str], min_words: int) -> list[int]:
    idx = [i for i, s in enumerate(sentences) if len(s.split()) >= min_words]
    return idx or list(range(len(sentences)))


def lead_k(text: str, k: int = 3, min_words: int = 4) -> str:
    """First k sentences (fragments shorter than `min_words` words are skipped)."""
    sentences = split_sentences(text)
    idx = _candidate_indices(sentences, min_words)[:k]
    return "\n".join(sentences[i] for i in idx)


def textrank_scores(sentences: list[str], damping: float = 0.85, max_iter: int = 100, tol: float = 1e-6) -> np.ndarray:
    """PageRank scores of sentences on a word-overlap similarity graph.

    sim(Si, Sj) = |Si ∩ Sj| / (log(1 + |Si|) + log(1 + |Sj|))   (stop-words removed)
    PR = (1 - d) / N + d * P^T PR,  P = row-normalised similarity matrix
    """
    n = len(sentences)
    if n == 0:
        return np.zeros(0)
    token_sets = [set(t for t in word_tokenize(s) if t not in STOPWORDS) for s in sentences]
    vocab: dict[str, int] = {}
    rows, cols = [], []
    for i, toks in enumerate(token_sets):
        for t in toks:
            rows.append(i)
            cols.append(vocab.setdefault(t, len(vocab)))
    if not vocab:
        return np.full(n, 1.0 / n)
    b = np.zeros((n, len(vocab)), dtype=np.float32)
    b[rows, cols] = 1.0
    overlap = b @ b.T                                   # |Si ∩ Sj|
    log_len = np.log1p(b.sum(axis=1))
    denom = log_len[:, None] + log_len[None, :]
    w = np.divide(overlap, denom, out=np.zeros_like(overlap), where=denom > 0)
    np.fill_diagonal(w, 0.0)
    out_weight = w.sum(axis=1)
    p = np.full((n, n), 1.0 / n, dtype=np.float64)       # dangling rows -> uniform jump
    has_out = out_weight > 0
    p[has_out] = w[has_out] / out_weight[has_out, None]
    scores = np.full(n, 1.0 / n)
    for _ in range(max_iter):
        new = (1.0 - damping) / n + damping * (p.T @ scores)
        converged = np.abs(new - scores).sum() < tol
        scores = new
        if converged:
            break
    return scores


def textrank(text: str, k: int = 3, min_words: int = 4, **kwargs) -> str:
    """Top-k TextRank sentences, returned in document order."""
    sentences = split_sentences(text)
    idx = _candidate_indices(sentences, min_words)
    if len(idx) <= k:
        return "\n".join(sentences[i] for i in idx)
    scores = textrank_scores([sentences[i] for i in idx], **kwargs)
    top = sorted(range(len(idx)), key=lambda j: (-scores[j], j))[:k]   # ties -> earlier sentence
    return "\n".join(sentences[idx[j]] for j in sorted(top))


def mmr_select(embeddings: np.ndarray, k: int, lambda_: float = 0.7) -> list[int]:
    """Maximal Marginal Relevance on L2-normalised embeddings, relevance = cosine to centroid."""
    n = len(embeddings)
    if n == 0:
        return []
    centroid = embeddings.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-12)
    relevance = embeddings @ centroid
    similarity = embeddings @ embeddings.T
    selected = [int(np.argmax(relevance))]
    while len(selected) < min(k, n):
        redundancy = similarity[:, selected].max(axis=1)
        score = lambda_ * relevance - (1.0 - lambda_) * redundancy
        score[selected] = -np.inf
        selected.append(int(np.argmax(score)))
    return selected


class EmbeddingExtractor:
    """Unsupervised extractive summarizer on top of a transformer sentence encoder.

    Default encoder: sentence-transformers/all-MiniLM-L6-v2 (22M params), mean pooling.
    """

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2", device: str | None = None,
                 batch_size: int = 256, max_length: int = 128, fp16: bool | None = None):
        import torch
        from transformers import AutoModel, AutoTokenizer

        self._torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).eval()
        if fp16 is None:
            fp16 = self.device.startswith("cuda")
        if fp16:
            self.model.half()
        self.model.to(self.device)
        self.batch_size, self.max_length = batch_size, max_length

    def encode(self, sentences: list[str]) -> np.ndarray:
        torch = self._torch
        order = np.argsort([-len(s) for s in sentences], kind="stable")   # length-sorted -> less padding
        dim = self.model.config.hidden_size
        out = np.zeros((len(sentences), dim), dtype=np.float32)
        for start in range(0, len(sentences), self.batch_size):
            batch_idx = order[start:start + self.batch_size]
            enc = self.tokenizer([sentences[i] for i in batch_idx], padding=True, truncation=True,
                                 max_length=self.max_length, return_tensors="pt").to(self.device)
            with torch.inference_mode():
                hidden = self.model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            pooled = torch.nn.functional.normalize(pooled.float(), dim=-1)
            out[batch_idx] = pooled.cpu().numpy()
        return out

    def summarize_many(self, texts: list[str], k: int = 3, lambda_: float = 0.7, min_words: int = 4,
                       docs_per_chunk: int = 2000, verbose: bool = False) -> list[str]:
        results: list[str] = []
        for c0 in range(0, len(texts), docs_per_chunk):
            docs, flat = [], []
            for text in texts[c0:c0 + docs_per_chunk]:
                sentences = split_sentences(text)
                idx = _candidate_indices(sentences, min_words)
                docs.append((sentences, idx, len(flat)))
                flat.extend(sentences[i] for i in idx)
            emb = self.encode(flat) if flat else np.zeros((0, 1), dtype=np.float32)
            for sentences, idx, offset in docs:
                if len(idx) <= k:
                    results.append("\n".join(sentences[i] for i in idx))
                    continue
                chosen = mmr_select(emb[offset:offset + len(idx)], k, lambda_)
                results.append("\n".join(sentences[idx[j]] for j in sorted(chosen)))
            if verbose:
                print(f"  embedding-MMR: {min(c0 + docs_per_chunk, len(texts))}/{len(texts)} documents")
        return results

    def summarize(self, text: str, k: int = 3, lambda_: float = 0.7) -> str:
        return self.summarize_many([text], k=k, lambda_=lambda_)[0]

In [ ]:
%%writefile {WORK_DIR}/src/metrics.py
"""Evaluation metrics for summarization.

* ROUGE-1/2/L/Lsum F1 (x100) via Google's `rouge-score` package with Porter stemming.
  Sentences are put on separate lines so that ROUGE-Lsum is the summary-level LCS used in papers.
* 95% bootstrap confidence intervals and paired bootstrap tests between two systems.
* Abstractiveness: % of summary n-grams that never occur in the source article.
* Length statistics.
"""
from __future__ import annotations

import numpy as np

from .preprocessing import split_sentences, word_tokenize

__all__ = ["ROUGE_TYPES", "rouge_per_sample", "bootstrap_ci", "paired_bootstrap", "novel_ngram_ratio",
           "length_stats", "evaluate_system"]

ROUGE_TYPES = ("rouge1", "rouge2", "rougeL", "rougeLsum")


def _one_sentence_per_line(text: str) -> str:
    return "\n".join(split_sentences(text))


def rouge_per_sample(predictions: list[str], references: list[str], use_stemmer: bool = True,
                     rouge_types=ROUGE_TYPES) -> dict[str, np.ndarray]:
    """Per-example ROUGE F1 scores (x100)."""
    from rouge_score import rouge_scorer

    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} predictions vs {len(references)} references")
    scorer = rouge_scorer.RougeScorer(list(rouge_types), use_stemmer=use_stemmer)
    scores = {t: np.zeros(len(predictions), dtype=np.float64) for t in rouge_types}
    for i, (pred, ref) in enumerate(zip(predictions, references)):
        s = scorer.score(_one_sentence_per_line(ref), _one_sentence_per_line(pred))   # (target, prediction)
        for t in rouge_types:
            scores[t][i] = 100.0 * s[t].fmeasure
    return scores


def bootstrap_ci(values, n_resamples: int = 1000, alpha: float = 0.05, seed: int = 0) -> tuple[float, float]:
    """Percentile bootstrap confidence interval of the mean."""
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    means = np.array([values[rng.integers(0, values.size, values.size)].mean() for _ in range(n_resamples)])
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def paired_bootstrap(a, b, n_resamples: int = 1000, seed: int = 0) -> dict:
    """Mean difference a-b with 95% CI and the fraction of resamples where a <= b (one-sided p-value)."""
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    if a.size == 0 or a.size != b.size:
        return {"diff": float("nan"), "ci": [float("nan"), float("nan")], "p_value": float("nan")}
    d = a - b
    rng = np.random.default_rng(seed)
    means = np.array([d[rng.integers(0, d.size, d.size)].mean() for _ in range(n_resamples)])
    lo, hi = np.percentile(means, [2.5, 97.5])
    return {"diff": round(float(d.mean()), 2), "ci": [round(float(lo), 2), round(float(hi), 2)],
            "p_value": round(float((means <= 0).mean()), 4)}


def _ngrams(tokens: list[str], n: int) -> set:
    return set(zip(*[tokens[i:] for i in range(n)]))


def novel_ngram_ratio(sources: list[str], summaries: list[str], ns=(1, 2, 3, 4)) -> dict[str, float]:
    """Average % of summary n-grams that do not appear in the source (0 = purely extractive)."""
    ratios = {n: [] for n in ns}
    for src, summ in zip(sources, summaries):
        src_tokens, summ_tokens = word_tokenize(src), word_tokenize(summ)
        for n in ns:
            summ_ngrams = _ngrams(summ_tokens, n)
            if summ_ngrams:
                ratios[n].append(len(summ_ngrams - _ngrams(src_tokens, n)) / len(summ_ngrams))
    return {str(n): (round(100.0 * float(np.mean(v)), 2) if v else float("nan")) for n, v in ratios.items()}


def length_stats(texts: list[str]) -> dict[str, float]:
    words = np.array([len(t.split()) for t in texts]) if texts else np.zeros(1)
    sents = np.array([len(split_sentences(t)) for t in texts]) if texts else np.zeros(1)
    return {"avg_words": round(float(words.mean()), 1), "median_words": float(np.median(words)),
            "avg_sentences": round(float(sents.mean()), 2)}


def evaluate_system(predictions: list[str], references: list[str], sources: list[str] | None = None,
                    novelty_idx=None, with_ci: bool = True) -> tuple[dict, dict]:
    """ROUGE (+CI), length and (optionally, on a subset) abstractiveness for one system."""
    per_sample = rouge_per_sample(predictions, references)
    result: dict = {"n": len(predictions)}
    for t, v in per_sample.items():
        result[t] = round(float(v.mean()), 2)
        if with_ci:
            lo, hi = bootstrap_ci(v)
            result[t + "_ci"] = [round(lo, 2), round(hi, 2)]
    result.update(length_stats(predictions))
    if sources is not None:
        idx = range(len(predictions)) if novelty_idx is None else novelty_idx
        result["novel_ngrams"] = novel_ngram_ratio([sources[i] for i in idx], [predictions[i] for i in idx])
    return result, per_sample

In [ ]:
%%writefile {WORK_DIR}/src/summarizer.py
"""Inference wrapper for fine-tuned seq2seq summarizers (BART / T5 / PEGASUS).

    from src.summarizer import Summarizer
    s = Summarizer("bart-base-cnn-summarizer")          # local folder or Hugging Face Hub id
    s.summarize(article)                                # one text -> str
    s.summarize([a1, a2], batch_size=16)               # many texts -> list[str]
    s.summarize_long(paper_text)                        # research papers / very long documents

Long documents are handled by hierarchical ("map-reduce") summarization: the text is split into
sentence-aligned chunks that fit the encoder, every chunk is summarized, the partial summaries are
concatenated and summarized again.
"""
from __future__ import annotations

import json
import os
import time

from .preprocessing import clean_article, split_sentences

__all__ = ["Summarizer", "DEFAULT_GENERATION", "chunk_sentences", "is_t5_like"]

# Decoding settings used for CNN/DailyMail (same as the reference BART-large-CNN configuration).
DEFAULT_GENERATION = dict(num_beams=4, max_new_tokens=142, min_new_tokens=56, length_penalty=2.0,
                          no_repeat_ngram_size=3, early_stopping=True)
T5_TYPES = {"t5", "mt5", "umt5", "longt5"}


def is_t5_like(model_type: str) -> bool:
    return model_type in T5_TYPES


def _split_long_sentence(sentence: str, count_tokens, max_tokens: int) -> list[str]:
    pieces, current = [], []
    for word in sentence.split():
        current.append(word)
        if len(current) > 1 and count_tokens(" ".join(current)) > max_tokens:
            current.pop()
            pieces.append(" ".join(current))
            current = [word]
    if current:
        pieces.append(" ".join(current))
    return pieces


def chunk_sentences(sentences: list[str], count_tokens, max_tokens: int, overlap: int = 1) -> list[list[str]]:
    """Greedy sentence packing into chunks of <= max_tokens tokens with `overlap` sentences of context."""
    max_tokens = max(2, int(max_tokens))
    units = []                                   # every unit costs n + 1 tokens (1 = separator margin)
    for s in sentences:
        n = count_tokens(s)
        if n + 1 > max_tokens:
            units.extend((p, count_tokens(p)) for p in _split_long_sentence(s, count_tokens, max_tokens - 1))
        else:
            units.append((s, n))
    chunks, current, current_len = [], [], 0
    for s, n in units:
        if current and current_len + n + 1 > max_tokens:
            chunks.append([u for u, _ in current])
            keep = current[-overlap:] if overlap > 0 else []
            keep_len = sum(k + 1 for _, k in keep)
            if keep_len + n + 1 > max_tokens:
                keep, keep_len = [], 0
            current, current_len = list(keep), keep_len
        current.append((s, n))
        current_len += n + 1
    if current:
        chunks.append([u for u, _ in current])
    return chunks


class Summarizer:
    def __init__(self, model_name_or_path: str, device: str | None = None, fp16: bool | None = None,
                 max_source_len: int | None = None, generation: dict | None = None):
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        self._torch = torch
        self.name = model_name_or_path
        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path).eval()
        meta_path = os.path.join(model_name_or_path, "summarizer_config.json")
        self.meta = json.load(open(meta_path)) if os.path.isfile(meta_path) else {}
        model_type = self.model.config.model_type
        self.prefix = self.meta.get("prefix", "summarize: " if is_t5_like(model_type) else "")
        self.clean_input = self.meta.get("clean_input", True)
        position_limit = getattr(self.model.config, "max_position_embeddings", None) or 1024
        self.max_source_len = int(max_source_len or self.meta.get("max_source_len") or min(position_limit, 1024))
        self.generation = dict(DEFAULT_GENERATION)
        self.generation.update(self.meta.get("generation", {}))
        self.generation.update(generation or {})
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if fp16 is None:
            fp16 = self.device.startswith("cuda") and not is_t5_like(model_type)
        if fp16:
            self.model.half()
        self.model.to(self.device)

    # ----------------------------------------------------------------------------------------- utils
    @property
    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.model.parameters())

    def prepare(self, text: str) -> str:
        return clean_article(text) if self.clean_input else " ".join(str(text).split())

    def count_tokens(self, text: str) -> int:
        return len(self.tokenizer(text, add_special_tokens=False)["input_ids"])

    def _resolve(self, overrides: dict) -> dict:
        gen = dict(self.generation)
        gen.update({k: v for k, v in overrides.items() if v is not None})
        if gen.get("max_new_tokens") is not None and gen.get("min_new_tokens") is not None:
            gen["min_new_tokens"] = max(0, min(gen["min_new_tokens"], gen["max_new_tokens"] - 1))
        return gen

    def _adaptive_lengths(self, n_input_tokens: int) -> dict:
        """Shorter inputs get proportionally shorter summaries (CNN/DM settings assume ~700 words)."""
        max_new = min(self.generation.get("max_new_tokens", 142), max(24, int(0.55 * n_input_tokens)))
        min_new = min(self.generation.get("min_new_tokens", 56), max(8, int(0.15 * n_input_tokens)))
        return {"max_new_tokens": max_new, "min_new_tokens": min(min_new, max_new - 1)}

    # ------------------------------------------------------------------------------------ generation
    def _generate(self, prepared: list[str], batch_size: int, gen: dict, progress_every: int = 0) -> list[str]:
        torch = self._torch
        order = sorted(range(len(prepared)), key=lambda i: -len(prepared[i]))
        outputs: list[str | None] = [None] * len(prepared)
        t0, n_batches = time.time(), (len(order) + batch_size - 1) // max(1, batch_size)
        for b, start in enumerate(range(0, len(order), batch_size), 1):
            idx = order[start:start + batch_size]
            enc = self.tokenizer([self.prefix + prepared[i] for i in idx], max_length=self.max_source_len,
                                 truncation=True, padding=True, return_tensors="pt").to(self.device)
            with torch.inference_mode():
                generated = self.model.generate(**enc, **gen)
            decoded = self.tokenizer.batch_decode(generated, skip_special_tokens=True)
            for i, text in zip(idx, decoded):
                outputs[i] = text.strip()
            if progress_every and (b % progress_every == 0 or b == n_batches):
                elapsed = time.time() - t0
                eta = elapsed / b * (n_batches - b)
                print(f"  generated {min(start + batch_size, len(order)):>6}/{len(order)} "
                      f"| {elapsed / 60:5.1f} min elapsed | ETA {eta / 60:5.1f} min", flush=True)
        return outputs  # type: ignore[return-value]

    def summarize(self, texts, batch_size: int = 16, adaptive_length: bool | None = None,
                  progress_every: int = 0, **generation_overrides):
        """Summarize one text (returns str) or a list of texts (returns list[str]).

        adaptive_length (default: True for a single text, False for lists) scales the summary length
        with the input length, which gives better results for short, non-news inputs.
        """
        single = isinstance(texts, str)
        items = [texts] if single else list(texts)
        if adaptive_length is None:
            adaptive_length = single
        prepared = [self.prepare(t) for t in items]
        if not adaptive_length:
            outs = self._generate(prepared, batch_size, self._resolve(generation_overrides), progress_every)
        else:
            outs = []
            for p in prepared:
                n_tokens = min(self.count_tokens(p), self.max_source_len)
                overrides = {**self._adaptive_lengths(n_tokens), **generation_overrides}
                outs.extend(self._generate([p], 1, self._resolve(overrides)))
        return outs[0] if single else outs

    def summarize_long(self, text: str, chunk_tokens: int | None = None, overlap_sentences: int = 1,
                       chunk_generation: dict | None = None, final_generation: dict | None = None,
                       max_rounds: int = 2, batch_size: int = 8) -> dict:
        """Hierarchical summarization for documents longer than the encoder window."""
        prepared = self.prepare(text)
        budget = (chunk_tokens or self.max_source_len) - self.count_tokens(self.prefix) - 8
        n_tokens = self.count_tokens(prepared)
        if n_tokens <= budget:
            return {"summary": self.summarize(prepared, **(final_generation or {})), "input_tokens": n_tokens,
                    "chunks": 1, "rounds": 0, "partial_summaries": []}
        current, rounds, partial_log, n_chunks_first = prepared, 0, [], None
        while rounds < max_rounds and self.count_tokens(current) > budget:
            chunks = chunk_sentences(split_sentences(current), self.count_tokens, budget, overlap_sentences)
            n_chunks_first = n_chunks_first or len(chunks)
            chunk_gen = chunk_generation or {"max_new_tokens": 96, "min_new_tokens": 24}
            partials = self.summarize([" ".join(c) for c in chunks], batch_size=batch_size,
                                      adaptive_length=False, **chunk_gen)
            partial_log.append(partials)
            current = " ".join(p.replace("\n", " ") for p in partials)
            rounds += 1
        final = self.summarize(current, adaptive_length=False, **(final_generation or {}))
        return {"summary": final, "input_tokens": n_tokens, "chunks": n_chunks_first, "rounds": rounds,
                "partial_summaries": partial_log}

In [ ]:
import importlib
import src, src.preprocessing, src.extractive, src.metrics, src.summarizer
for _m in (src, src.preprocessing, src.extractive, src.metrics, src.summarizer):
    importlib.reload(_m)
from src.preprocessing import clean_article, clean_summary, split_sentences, count_words
from src.extractive import lead_k, textrank, EmbeddingExtractor
from src.metrics import evaluate_system, paired_bootstrap, novel_ngram_ratio, ROUGE_TYPES
from src.summarizer import Summarizer, is_t5_like
print("src package ready, version", src.__version__)

## 2. Data
The Kaggle dataset is the non-anonymised CNN/DailyMail corpus (version 3.0.0): columns `id`, `article`, `highlights` (the bullet-point highlights are the reference summaries). The CSV files are discovered automatically under `/kaggle/input`. Rows with a missing article/summary or an article shorter than 20 words are dropped.

Cleaning removes scraping boilerplate from articles (e.g. `By . Daily Mail Reporter . PUBLISHED: . 14:11 EST, 25 October 2013 .` or `LONDON, England (CNN) --`). Reference summaries are only whitespace-normalised (one highlight per line).

In [ ]:
ARTICLE_COLS = ("article", "text", "document", "story", "content")
SUMMARY_COLS = ("highlights", "summary", "abstract", "headline", "summaries")


def discover_csvs(root):
    csvs = sorted(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True))
    if not csvs:
        raise FileNotFoundError(f"No CSV files under {root!r}. On Kaggle: 'Add Input' -> "
                                "'newspaper-text-summarization-cnn-dailymail' (gowrishankarp).")
    pool = [p for p in csvs if re.search(r"cnn|dailymail|daily_mail", p, re.I)] or csvs

    def pick(pattern):
        hits = [p for p in pool if re.search(pattern, os.path.basename(p), re.I)]
        return max(hits, key=os.path.getsize) if hits else None
    return {"train": pick(r"train"), "validation": pick(r"val|dev"), "test": pick(r"test")}


def load_split(path):
    df = pd.read_csv(path)
    cols = {str(c).lower().strip(): c for c in df.columns}
    a_col = next((cols[c] for c in ARTICLE_COLS if c in cols), None)
    s_col = next((cols[c] for c in SUMMARY_COLS if c in cols), None)
    if a_col is None or s_col is None:
        raise ValueError(f"{path}: cannot find article/summary columns in {list(df.columns)}")
    ids = df[cols["id"]].astype(str).to_numpy() if "id" in cols else np.arange(len(df)).astype(str)
    out = pd.DataFrame({"id": ids, "article": df[a_col].to_numpy(), "highlights": df[s_col].to_numpy()})
    n_raw = len(out)
    out = out.dropna(subset=["article", "highlights"])
    out = out[out["article"].astype(str).str.count(r"\S+") >= 20]
    out = out[out["highlights"].astype(str).str.strip().str.len() > 0].reset_index(drop=True)
    out["article"], out["highlights"] = out["article"].astype(str), out["highlights"].astype(str)
    return out, n_raw - len(out)


DATA_FILES = discover_csvs(DATA_DIR)
print(json.dumps(DATA_FILES, indent=1))
frames, DROPPED = {}, {}
for split, path in DATA_FILES.items():
    if path:
        frames[split], DROPPED[split] = load_split(path)
if "train" not in frames:
    raise FileNotFoundError("No training CSV found (file name must contain 'train').")
if "test" not in frames or "validation" not in frames:            # single-file datasets -> 90/5/5
    full = frames["train"].sample(frac=1.0, random_state=CFG["seed"]).reset_index(drop=True)
    n_hold = max(1, int(0.05 * len(full)))
    frames.setdefault("test", full.iloc[:n_hold].reset_index(drop=True))
    frames.setdefault("validation", full.iloc[n_hold:2 * n_hold].reset_index(drop=True))
    frames["train"] = full.iloc[2 * n_hold:].reset_index(drop=True)
SPLIT_SIZES = {k: len(v) for k, v in frames.items()}


def subsample(df, n):
    return df if (n is None or n >= len(df)) else df.sample(n=n, random_state=CFG["seed"]).reset_index(drop=True)


train_df = subsample(frames["train"], CFG["train_samples"])
val_df = subsample(frames["validation"], CFG["val_samples"])
test_df = subsample(frames["test"], CFG["test_samples"])
del frames
gc.collect()
test_df["article_clean"] = test_df["article"].map(clean_article)
test_df["reference"] = test_df["highlights"].map(clean_summary)
print("rows per split:", SPLIT_SIZES, "| dropped:", DROPPED)
print(f"used -> train {len(train_df):,} | validation (eval loss) {len(val_df):,} | test {len(test_df):,}")

In [ ]:
from transformers import AutoConfig, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
MODEL_TYPE = AutoConfig.from_pretrained(CFG["model_name"]).model_type
T5_LIKE = is_t5_like(MODEL_TYPE)
PREFIX = "summarize: " if T5_LIKE else ""

eda = train_df.sample(n=min(20000, len(train_df)), random_state=CFG["seed"])
eda_art = eda["article"].map(clean_article)
eda_sum = eda["highlights"].map(clean_summary)
art_words = eda_art.str.split().str.len().to_numpy()
sum_words = eda_sum.str.split().str.len().to_numpy()
sum_sents = eda_sum.map(lambda s: len(split_sentences(s))).to_numpy()
tok_lens = np.array([len(x) for x in tokenizer(eda_art.iloc[:2000].tolist())["input_ids"]])
_nov = min(CFG["novelty_samples"], len(test_df))
DATA_STATS = dict(
    eda_sample=len(eda),
    article_words_mean=round(float(art_words.mean()), 1), article_words_median=float(np.median(art_words)),
    article_words_p95=float(np.percentile(art_words, 95)),
    summary_words_mean=round(float(sum_words.mean()), 1), summary_sentences_mean=round(float(sum_sents.mean()), 2),
    compression_ratio=round(float(np.mean(art_words / np.maximum(sum_words, 1))), 1),
    article_tokens_mean=round(float(tok_lens.mean()), 1),
    pct_articles_over_max_source_len=round(100 * float((tok_lens > CFG["max_source_len"]).mean()), 1),
    reference_novel_ngrams=novel_ngram_ratio(test_df["article_clean"].iloc[:_nov].tolist(),
                                             test_df["reference"].iloc[:_nov].tolist()),
)
CLEAN_EXAMPLE = None
for raw in test_df["article"].head(500):
    flat = " ".join(str(raw).split())
    cleaned = clean_article(raw)
    if cleaned[:60] != flat[:60]:
        CLEAN_EXAMPLE = {"before": flat[:250], "after": cleaned[:180]}
        break
print(json.dumps(DATA_STATS, indent=1))
print("cleaning example:", json.dumps(CLEAN_EXAMPLE, indent=1, ensure_ascii=False))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(np.clip(art_words, 0, 2000), bins=60, color="#4C72B0")
ax[0].set_title("Article length (words)")
ax[1].hist(np.clip(tok_lens, 0, 2500), bins=60, color="#55A868")
ax[1].axvline(CFG["max_source_len"], color="red", ls="--", label=f"max_source_len = {CFG['max_source_len']}")
ax[1].set_title(f"Article length ({MODEL_TAG} tokens)")
ax[1].legend()
ax[2].hist(np.clip(sum_words, 0, 150), bins=50, color="#C44E52")
ax[2].set_title("Reference summary length (words)")
for a in ax:
    a.set_ylabel("articles")
fig.suptitle(f"CNN/DailyMail training set (random sample of {len(eda):,} articles)")
fig.tight_layout()
FIGURES["lengths"] = "results/figures/fig_lengths.png"
fig.savefig(os.path.join(WORK_DIR, FIGURES["lengths"]), dpi=150)
plt.show()

## 3. Extractive baselines
* **LEAD-3** — the first three sentences. News is written as an *inverted pyramid*, so this is a very strong baseline on CNN/DailyMail.
* **TextRank** (Mihalcea & Tarau, 2004) — PageRank over a sentence graph, edge weight `|Si ∩ Sj| / (log|Si| + log|Sj|)`; top-3 sentences in document order.
* **MiniLM embedding + MMR** — sentences embedded with a transformer encoder (`all-MiniLM-L6-v2`, mean pooling); relevance = cosine similarity to the document centroid, diversity via Maximal Marginal Relevance (λ = 0.7).

ROUGE-1/2/L/Lsum F1 (× 100, Porter stemming) with 95% bootstrap confidence intervals. **ROUGE-Lsum is the summary-level ROUGE-L that papers report as "ROUGE-L" on CNN/DailyMail.**

In [ ]:
rng = np.random.default_rng(CFG["seed"])
N_TEST = len(test_df)
NOVELTY_IDX = np.sort(rng.choice(N_TEST, size=min(CFG["novelty_samples"], N_TEST), replace=False))
TEST_ARTICLES = test_df["article_clean"].tolist()
REFS = test_df["reference"].tolist()
PREDS, PER_SAMPLE, RESULTS, TIMING = {}, {}, {}, {}
DISPLAY_NAMES = {"lead3": "LEAD-3", "textrank": "TextRank", "embed_mmr": "MiniLM embedding + MMR",
                 "zero_shot": f"{MODEL_TAG} zero-shot (no fine-tuning)", "finetuned": f"{MODEL_TAG} fine-tuned (ours)"}


def run_system(name, fn):
    t0 = time.time()
    PREDS[name] = fn()
    TIMING[name] = round(time.time() - t0, 1)
    RESULTS[name], PER_SAMPLE[name] = evaluate_system(PREDS[name], REFS, TEST_ARTICLES, NOVELTY_IDX)
    r = RESULTS[name]
    print(f"{DISPLAY_NAMES.get(name, name):<34} R1 {r['rouge1']:6.2f} | R2 {r['rouge2']:6.2f} | RL {r['rougeL']:6.2f} "
          f"| RLsum {r['rougeLsum']:6.2f} | {TIMING[name]:.0f}s", flush=True)


def results_frame(results):
    rows = []
    for k, r in results.items():
        rows.append({"system": DISPLAY_NAMES.get(k, k), "n": r["n"], "ROUGE-1": r["rouge1"], "ROUGE-2": r["rouge2"],
                     "ROUGE-L": r["rougeL"], "ROUGE-Lsum": r["rougeLsum"], "avg words": r["avg_words"],
                     "novel 2-grams %": (r.get("novel_ngrams") or {}).get("2")})
    return pd.DataFrame(rows).set_index("system")


run_system("lead3", lambda: [lead_k(a, 3) for a in TEST_ARTICLES])
run_system("textrank", lambda: [textrank(a, 3) for a in TEST_ARTICLES])
if CFG["run_embed_extractive"]:
    try:
        _ext = EmbeddingExtractor(CFG["embed_model"])
        run_system("embed_mmr", lambda: _ext.summarize_many(TEST_ARTICLES, k=3, verbose=True))
        del _ext
    except Exception as e:                                   # e.g. internet off -> keep going
        print("embedding baseline skipped:", repr(e)[:300])
    gc.collect()
    torch.cuda.empty_cache()
display(results_frame(RESULTS))

## 4. Fine-tuning BART-base
* **Model**: `facebook/bart-base` — 6+6-layer encoder–decoder transformer (≈140 M parameters) pre-trained as a denoising autoencoder.
* **Input / target**: cleaned article truncated to 512 tokens → highlights joined into one paragraph (≤ 128 tokens).
* **Loss**: token-level cross-entropy with label smoothing ε = 0.1 (teacher forcing).
* **Optimiser**: AdamW (lr 5e-5, weight decay 0.01, none on biases/LayerNorm), gradient clipping 1.0, fp16 mixed precision, effective batch 32 (2 × T4 via DataParallel).
* **Time-budgeted learning-rate schedule** (so the run always finishes inside Kaggle's session limit, with a properly annealed LR):

$$\text{lr}(s) = \text{lr}_{\max}\cdot\min\!\Big(1, \frac{s+1}{W}\Big)\cdot\Big(1-\max\big(\tfrac{s}{S},\ \tfrac{t}{T}\big)\Big)$$

with step *s*, warm-up *W*, planned steps *S* (= `epochs` × steps/epoch), elapsed training time *t* and time budget *T* (`train_hours`). Training stops when either fraction reaches 1.

In [ ]:
from datasets import Dataset


def to_target(summary):
    """Highlights -> one paragraph of sentences (each ends with punctuation)."""
    lines = [l.strip() for l in clean_summary(summary).split("\n") if l.strip()]
    return " ".join(l if l[-1] in ".!?'\"" else l + "." for l in lines)


def preprocess(batch):
    sources = [PREFIX + clean_article(a) for a in batch["article"]]
    targets = [to_target(s) for s in batch["highlights"]]
    enc = tokenizer(sources, max_length=CFG["max_source_len"], truncation=True)
    enc["labels"] = tokenizer(text_target=targets, max_length=CFG["max_target_len"], truncation=True)["input_ids"]
    return enc


def tokenize(df, name):
    ds = Dataset.from_pandas(df[["article", "highlights"]].reset_index(drop=True), preserve_index=False)
    nproc = NUM_PROC if len(df) > 5000 else None
    print(f"tokenizing {name}: {len(df):,} examples with {nproc or 1} process(es) ...", flush=True)
    try:
        return ds.map(preprocess, batched=True, batch_size=1000, num_proc=nproc, remove_columns=ds.column_names,
                      desc=f"tokenize {name}")
    except Exception as e:                      # safety net: parallel map failed -> single process (~4x slower)
        if not nproc:
            raise
        print(f"parallel tokenization failed ({repr(e)[:200]}) -> retrying in one process", flush=True)
        return ds.map(preprocess, batched=True, batch_size=1000, remove_columns=ds.column_names,
                      desc=f"tokenize {name}")


t0 = time.time()
train_ds = tokenize(train_df, "train")
val_ds = tokenize(val_df, "validation")
_lab = np.array([len(x) for x in train_ds.select(range(min(5000, len(train_ds))))["labels"]])
print(f"tokenized train {len(train_ds):,} / validation {len(val_ds):,} in {time.time() - t0:.0f}s")
print(f"target length: mean {_lab.mean():.1f} tokens, {100 * (_lab >= CFG['max_target_len']).mean():.2f}% truncated")
print("example target:", tokenizer.decode(train_ds[0]["labels"], skip_special_tokens=True)[:300])

In [ ]:
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          TrainerCallback)

model = AutoModelForSeq2SeqLM.from_pretrained(CFG["model_name"])
USE_FP16 = bool(CFG["fp16"]) and N_GPU > 0 and not T5_LIKE          # T5 overflows in fp16
GEN_KW = dict(num_beams=CFG["num_beams"], max_new_tokens=CFG["max_new_tokens"], min_new_tokens=CFG["min_new_tokens"],
              length_penalty=CFG["length_penalty"], no_repeat_ngram_size=CFG["no_repeat_ngram_size"], early_stopping=True)
model.generation_config.update(**GEN_KW)
N_PARAMS = sum(p.numel() for p in model.parameters())

n_dev = max(1, N_GPU)
GRAD_ACCUM = max(1, CFG["effective_batch"] // (CFG["per_device_batch"] * n_dev))
EFFECTIVE_BATCH = CFG["per_device_batch"] * n_dev * GRAD_ACCUM
STEPS_PER_EPOCH = max(1, math.ceil(len(train_ds) / (CFG["per_device_batch"] * n_dev) / GRAD_ACCUM))
TOTAL_STEPS = max(1, int(round(STEPS_PER_EPOCH * CFG["epochs"])))
WARMUP = max(1, min(CFG["warmup_steps"], TOTAL_STEPS // 10))
EVAL_EVERY = max(1, min(CFG["eval_every"], TOTAL_STEPS))


def param_groups(model, weight_decay):
    """AdamW parameter groups: no weight decay for biases and normalisation weights."""
    no_decay = set()
    for mod_name, mod in model.named_modules():
        if isinstance(mod, torch.nn.LayerNorm) or "norm" in type(mod).__name__.lower():
            no_decay.update(f"{mod_name}.{p}" if mod_name else p for p, _ in mod.named_parameters(recurse=False))
    decay, nodecay = [], []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (nodecay if (n in no_decay or n.endswith("bias")) else decay).append(p)
    return [{"params": decay, "weight_decay": weight_decay}, {"params": nodecay, "weight_decay": 0.0}]


if N_GPU:
    model.to("cuda")
optimizer = torch.optim.AdamW(param_groups(model, CFG["weight_decay"]), lr=CFG["lr"], betas=(0.9, 0.999), eps=1e-8)


class TimeBudget:
    def __init__(self, hours):
        self.seconds, self.t0 = max(1.0, float(hours) * 3600.0), None

    def start(self):
        self.t0 = time.time()

    def elapsed(self):
        return 0.0 if self.t0 is None else time.time() - self.t0

    def fraction(self):
        return self.elapsed() / self.seconds


BUDGET = TimeBudget(CFG["train_hours"])


def lr_multiplier(step):
    """Linear warm-up, then linear decay driven by max(step progress, wall-clock progress)."""
    warm = min(1.0, (step + 1) / WARMUP)
    progress = max(step / TOTAL_STEPS, BUDGET.fraction())
    return warm * max(0.0, 1.0 - progress)


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)


class TimeBudgetCallback(TrainerCallback):
    def __init__(self, budget):
        self.budget, self.stopped_early = budget, False

    def on_train_begin(self, args, state, control, **kwargs):
        self.budget.start()

    def on_step_end(self, args, state, control, **kwargs):
        if self.budget.fraction() >= 1.0 and state.global_step < state.max_steps:
            control.should_training_stop = True
            self.stopped_early = True
            print(f"time budget ({CFG['train_hours']} h) reached at step {state.global_step}/{state.max_steps} -> stop")
        return control


class LogPrinter(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        logs, el = logs or {}, BUDGET.elapsed()
        if "loss" in logs:
            frac = max(state.global_step / max(1, state.max_steps), BUDGET.fraction())
            eta = el / frac - el if frac > 0 else float("nan")
            print(f"step {state.global_step:>6}/{state.max_steps} | epoch {state.epoch or 0:5.2f} | loss {logs['loss']:.4f} "
                  f"| lr {logs.get('learning_rate', float('nan')):.2e} | {el / 60:6.1f} min | ETA {eta / 60:6.1f} min",
                  flush=True)
        if "eval_loss" in logs:
            print(f"   >> step {state.global_step}: eval_loss {logs['eval_loss']:.4f}", flush=True)


def make_training_args(**kw):
    """Seq2SeqTrainingArguments that work across transformers 4.4x-5.x (renamed / removed fields)."""
    fields = {f.name for f in dataclasses.fields(Seq2SeqTrainingArguments)}
    if "eval_strategy" in kw and "eval_strategy" not in fields:
        kw["evaluation_strategy"] = kw.pop("eval_strategy")
    ignored = sorted(k for k in kw if k not in fields)
    if ignored:
        print("TrainingArguments: ignoring unsupported", ignored)
    return Seq2SeqTrainingArguments(**{k: v for k, v in kw.items() if k in fields})


training_args = make_training_args(
    output_dir=CKPT_DIR, per_device_train_batch_size=CFG["per_device_batch"],
    per_device_eval_batch_size=CFG["per_device_batch"] * 2, gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=TOTAL_STEPS, learning_rate=CFG["lr"], weight_decay=CFG["weight_decay"], max_grad_norm=CFG["max_grad_norm"],
    label_smoothing_factor=CFG["label_smoothing"], fp16=USE_FP16, eval_strategy="steps", eval_steps=EVAL_EVERY,
    eval_on_start=True, logging_strategy="steps", logging_steps=CFG["log_every"], logging_first_step=True,
    save_strategy="no", report_to="none", disable_tqdm=True, predict_with_generate=False,
    dataloader_num_workers=NUM_WORKERS, seed=CFG["seed"], remove_unused_columns=True)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100,
                                  pad_to_multiple_of=8 if USE_FP16 else None)
budget_cb = TimeBudgetCallback(BUDGET)
trainer_kw = dict(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator,
                  optimizers=(optimizer, scheduler), callbacks=[budget_cb, LogPrinter()])
_sig = inspect.signature(Seq2SeqTrainer.__init__).parameters
trainer_kw["processing_class" if "processing_class" in _sig else "tokenizer"] = tokenizer
trainer = Seq2SeqTrainer(**trainer_kw)
try:
    from transformers.trainer_callback import PrinterCallback
    trainer.remove_callback(PrinterCallback)                 # LogPrinter replaces the raw dict printer
except Exception:
    pass

print(f"model {CFG['model_name']}: {N_PARAMS / 1e6:.1f} M parameters | fp16 {USE_FP16} | GPUs {N_GPU}")
print(f"batch {CFG['per_device_batch']} x {n_dev} device(s) x {GRAD_ACCUM} accumulation = {EFFECTIVE_BATCH}")
print(f"{STEPS_PER_EPOCH:,} steps/epoch | planned {TOTAL_STEPS:,} steps ({CFG['epochs']} epochs) | warm-up {WARMUP} "
      f"| time budget {CFG['train_hours']} h | eval every {EVAL_EVERY} steps")

In [ ]:
t_train = time.time()
train_output = trainer.train()
TRAIN_SECONDS = time.time() - t_train
final_eval = trainer.evaluate()
state = trainer.state
TRAIN_INFO = dict(
    params_million=round(N_PARAMS / 1e6, 1), n_gpu=N_GPU, gpu=ENV["gpu"], fp16=USE_FP16,
    per_device_batch=CFG["per_device_batch"], grad_accum=GRAD_ACCUM, effective_batch=EFFECTIVE_BATCH,
    epochs_planned=CFG["epochs"], steps_per_epoch=STEPS_PER_EPOCH, total_steps_planned=TOTAL_STEPS,
    steps_completed=state.global_step, epochs_completed=round(state.global_step / STEPS_PER_EPOCH, 3),
    samples_seen=state.global_step * EFFECTIVE_BATCH, train_examples=len(train_ds), val_examples=len(val_ds),
    warmup_steps=WARMUP, lr=CFG["lr"], weight_decay=CFG["weight_decay"], label_smoothing=CFG["label_smoothing"],
    max_grad_norm=CFG["max_grad_norm"], max_source_len=CFG["max_source_len"], max_target_len=CFG["max_target_len"],
    time_budget_hours=CFG["train_hours"], stopped_by_time_budget=budget_cb.stopped_early,
    train_runtime_hours=round(TRAIN_SECONDS / 3600, 3), final_train_loss=round(float(train_output.training_loss), 4),
    final_eval_loss=round(float(final_eval["eval_loss"]), 4),
    throughput_samples_per_s=round(state.global_step * EFFECTIVE_BATCH / max(TRAIN_SECONDS, 1e-9), 2),
    log_history=state.log_history)
print(json.dumps({k: v for k, v in TRAIN_INFO.items() if k != "log_history"}, indent=1))

## 5. Save the model
Saved in the standard **Hugging Face format** (`model.safetensors` + tokenizer + `generation_config.json`) plus `summarizer_config.json` (input prefix, max input length, decoding settings) used by `src/summarizer.py`. The folder is also zipped for a one-click download. (Why not a `.pkl`? See the FAQ in the README: pickles execute code when loaded and break across library versions; safetensors is the portable, safe format.)

In [ ]:
def zip_dir(folder):
    path = folder + ".zip"
    with zipfile.ZipFile(path, "w", zipfile.ZIP_STORED) as zf:     # safetensors do not compress; STORED is fast
        for root, _, files in os.walk(folder):
            for f in sorted(files):
                full = os.path.join(root, f)
                zf.write(full, os.path.relpath(full, os.path.dirname(folder)))
    return path


os.makedirs(MODEL_DIR, exist_ok=True)
trainer.model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
SUMMARIZER_META = dict(
    base_model=CFG["model_name"], prefix=PREFIX, clean_input=True, max_source_len=CFG["max_source_len"],
    max_target_len=CFG["max_target_len"], generation=GEN_KW, run_mode=RUN_MODE,
    dataset="CNN/DailyMail 3.0.0 (Kaggle: gowrishankarp/newspaper-text-summarization-cnn-dailymail)",
    steps=TRAIN_INFO["steps_completed"], samples_seen=TRAIN_INFO["samples_seen"],
    train_hours=TRAIN_INFO["train_runtime_hours"], created=datetime.datetime.now().isoformat(timespec="seconds"),
    project="https://github.com/ksidharth8/text-summarization-transformers")
with open(os.path.join(MODEL_DIR, "summarizer_config.json"), "w") as f:
    json.dump(SUMMARIZER_META, f, indent=2)
MODEL_ZIP = zip_dir(MODEL_DIR)
for f in sorted(os.listdir(MODEL_DIR)):
    print(f"  {f:<28} {os.path.getsize(os.path.join(MODEL_DIR, f)) / 1e6:9.2f} MB")
print(f"saved -> {MODEL_DIR}\nzip   -> {MODEL_ZIP} ({os.path.getsize(MODEL_ZIP) / 1e6:.0f} MB)")

In [ ]:
hist = pd.DataFrame(TRAIN_INFO["log_history"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
if "loss" in hist:
    tr = hist.dropna(subset=["loss"])
    ax[0].plot(tr["step"], tr["loss"], label="train loss", color="#4C72B0", alpha=0.8)
if "eval_loss" in hist:
    ev = hist.dropna(subset=["eval_loss"])
    ax[0].plot(ev["step"], ev["eval_loss"], "o-", label="validation loss", color="#C44E52")
ax[0].set_xlabel("optimizer step")
ax[0].set_ylabel("label-smoothed cross-entropy")
ax[0].set_title("Loss")
ax[0].legend()
if "learning_rate" in hist:
    lr_ = hist.dropna(subset=["learning_rate"])
    ax[1].plot(lr_["step"], lr_["learning_rate"], color="#55A868")
ax[1].set_xlabel("optimizer step")
ax[1].set_title("Learning rate (time-budgeted schedule)")
fig.tight_layout()
FIGURES["training_curves"] = "results/figures/fig_training_curves.png"
fig.savefig(os.path.join(WORK_DIR, FIGURES["training_curves"]), dpi=150)
plt.show()

del trainer, model, optimizer, scheduler, train_ds, val_ds, train_df
gc.collect()
torch.cuda.empty_cache()

## 6. Evaluation on the test set
The **saved** model is reloaded (this tests exactly the artifact you download) and summarizes every test article with beam search (4 beams, 56–142 new tokens, length penalty 2.0, no repeated trigrams). For comparison, the *untouched* pre-trained model is run zero-shot on a random subset.

In [ ]:
summarizer = Summarizer(MODEL_DIR)
print(f"loaded {MODEL_DIR} on {summarizer.device} | max_source_len {summarizer.max_source_len} | {summarizer.generation}")
_n_batches = math.ceil(N_TEST / CFG["gen_batch"])
run_system("finetuned", lambda: summarizer.summarize(TEST_ARTICLES, batch_size=CFG["gen_batch"], adaptive_length=False,
                                                     progress_every=max(1, _n_batches // 20)))
GEN_INFO = dict(GEN_KW, batch_size=CFG["gen_batch"], seconds=TIMING["finetuned"],
                docs_per_second=round(N_TEST / max(TIMING["finetuned"], 1e-9), 2))

In [ ]:
ZS_IDX = np.sort(rng.choice(N_TEST, size=min(CFG["zero_shot_samples"], N_TEST), replace=False))
ZS_PREDS = []
try:
    _zs = Summarizer(CFG["model_name"], max_source_len=CFG["max_source_len"], generation=GEN_KW)
    ZS_PREDS = _zs.summarize([TEST_ARTICLES[i] for i in ZS_IDX], batch_size=CFG["gen_batch"], adaptive_length=False)
    RESULTS["zero_shot"], PER_SAMPLE["zero_shot"] = evaluate_system(
        ZS_PREDS, [REFS[i] for i in ZS_IDX], [TEST_ARTICLES[i] for i in ZS_IDX])
    del _zs
except Exception as e:
    print("zero-shot run skipped:", repr(e)[:300])
gc.collect()
torch.cuda.empty_cache()
ZS_SUBSET = {k: {t: round(float(PER_SAMPLE[k][t][ZS_IDX].mean()), 2) for t in ROUGE_TYPES}
             for k in ("lead3", "finetuned") if k in PER_SAMPLE}
if "zero_shot" in ZS_SUBSET:
    ZS_SUBSET["zero_shot"] = {t: RESULTS["zero_shot"][t] for t in ROUGE_TYPES}
print(f"same {len(ZS_IDX)}-article subset:")
display(pd.DataFrame(ZS_SUBSET).T.rename(index=DISPLAY_NAMES))

In [ ]:
SIGNIFICANCE = {}
for base in ("lead3", "textrank", "embed_mmr"):
    if base in PER_SAMPLE:
        SIGNIFICANCE[f"finetuned_vs_{base}"] = {t: paired_bootstrap(PER_SAMPLE["finetuned"][t], PER_SAMPLE[base][t])
                                                for t in ROUGE_TYPES}
display(Markdown(f"### Main results — CNN/DailyMail test set (n = {N_TEST:,})"))
display(results_frame(RESULTS))
for name, sig in SIGNIFICANCE.items():
    print(f"{name}: " + " | ".join(f"{t} {v['diff']:+.2f} [{v['ci'][0]:+.2f}, {v['ci'][1]:+.2f}] p={v['p_value']}"
                                 for t, v in sig.items()))

# ROUGE-1 by article length (quartiles): shows the effect of truncating long inputs to max_source_len
wc = np.array([len(a.split()) for a in TEST_ARTICLES])
edges = np.unique(np.quantile(wc, [0, 0.25, 0.5, 0.75, 1.0]))
buckets = np.clip(np.searchsorted(edges, wc, side="right") - 1, 0, max(0, len(edges) - 2))
BY_LENGTH = []
for b in range(max(0, len(edges) - 1)):
    m = buckets == b
    if m.sum() == 0:
        continue
    row = {"bucket": f"{int(edges[b])}-{int(edges[b + 1])} words", "n": int(m.sum())}
    for k in ("lead3", "textrank", "finetuned"):
        if k in PER_SAMPLE:
            row[k] = round(float(PER_SAMPLE[k]["rouge1"][m].mean()), 2)
    BY_LENGTH.append(row)
display(pd.DataFrame(BY_LENGTH))

In [ ]:
SYSTEMS = [k for k in ("lead3", "textrank", "embed_mmr", "zero_shot", "finetuned") if k in RESULTS]
COLORS = {"lead3": "#8C8C8C", "textrank": "#4C72B0", "embed_mmr": "#55A868", "zero_shot": "#DD8452", "finetuned": "#C44E52"}
fig, ax = plt.subplots(figsize=(12, 4.5))
mets = [("rouge1", "ROUGE-1"), ("rouge2", "ROUGE-2"), ("rougeLsum", "ROUGE-Lsum")]
w, x = 0.8 / len(SYSTEMS), np.arange(len(mets))
for i, k in enumerate(SYSTEMS):
    vals = [RESULTS[k][m] for m, _ in mets]
    err = np.array([[RESULTS[k][m] - RESULTS[k][m + "_ci"][0], RESULTS[k][m + "_ci"][1] - RESULTS[k][m]] for m, _ in mets]).T
    bars = ax.bar(x + (i - (len(SYSTEMS) - 1) / 2) * w, vals, w, yerr=np.abs(err), capsize=3, color=COLORS[k],
                  label=f"{DISPLAY_NAMES[k]} (n={RESULTS[k]['n']:,})")
    ax.bar_label(bars, fmt="%.1f", fontsize=7, padding=2)
ax.set_xticks(x, [n for _, n in mets])
ax.set_ylabel("F1 × 100")
ax.set_ylim(0, max(RESULTS[k][m + "_ci"][1] for k in SYSTEMS for m, _ in mets) * 1.1)
ax.set_title("ROUGE on the CNN/DailyMail test set (error bars: 95% bootstrap CI)")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False)
fig.tight_layout()
FIGURES["rouge"] = "results/figures/fig_rouge_comparison.png"
fig.savefig(os.path.join(WORK_DIR, FIGURES["rouge"]), dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ns = ["1", "2", "3", "4"]
ax[0].plot(ns, [DATA_STATS["reference_novel_ngrams"][n] for n in ns], "k--o", label="reference summaries")
for k in SYSTEMS:
    if RESULTS[k].get("novel_ngrams"):
        ax[0].plot(ns, [RESULTS[k]["novel_ngrams"][n] for n in ns], "o-", color=COLORS[k], label=DISPLAY_NAMES[k])
ax[0].set_xlabel("n-gram order")
ax[0].set_ylabel("% of summary n-grams not in the article")
ax[0].set_title("Abstractiveness (novel n-grams)")
ax[0].legend(fontsize=8)
if BY_LENGTH:
    bl = pd.DataFrame(BY_LENGTH)
    xs = np.arange(len(bl))
    cols = [k for k in ("lead3", "textrank", "finetuned") if k in bl]
    for i, k in enumerate(cols):
        bars = ax[1].bar(xs + (i - (len(cols) - 1) / 2) * 0.27, bl[k], 0.27, color=COLORS[k], label=DISPLAY_NAMES[k])
        ax[1].bar_label(bars, fmt="%.1f", fontsize=7, padding=2)
    ax[1].set_xticks(xs, bl["bucket"], fontsize=8)
    ax[1].set_ylim(0, float(bl[cols].to_numpy().max()) * 1.12)
    ax[1].set_ylabel("ROUGE-1")
    ax[1].set_title("ROUGE-1 by article length (test quartiles)")
    ax[1].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
fig.tight_layout()
FIGURES["novelty_length"] = "results/figures/fig_novelty_and_length.png"
fig.savefig(os.path.join(WORK_DIR, FIGURES["novelty_length"]), dpi=150, bbox_inches="tight")
plt.show()

## 7. Qualitative examples and custom test cases
Random test articles with the reference and system summaries (per-example ROUGE in brackets), then texts that are **not** from CNN/DailyMail (`test_cases/`), including a research-paper-style document longer than the 512-token input window, which is summarized hierarchically (chunk → partial summaries → final summary).

In [ ]:
EX_IDX = rng.choice(N_TEST, size=min(CFG["n_examples"], N_TEST), replace=False)
SAMPLES, md_lines = [], []
for n, i in enumerate(map(int, EX_IDX), 1):
    item = {"id": str(test_df["id"].iloc[i]), "article_excerpt": " ".join(TEST_ARTICLES[i].split()[:120]) + " ...",
            "reference": REFS[i]}
    md_lines.append(f"#### Example {n} (id `{item['id']}`)\n**Article (first 120 words):** {item['article_excerpt']}\n\n"
                    f"**Reference:** {REFS[i].replace(chr(10), ' ')}\n")
    for k in ("lead3", "textrank", "finetuned"):
        if k in PREDS:
            sc = {t: round(float(PER_SAMPLE[k][t][i]), 1) for t in ("rouge1", "rouge2", "rougeLsum")}
            item[k] = dict(summary=PREDS[k][i], **sc)
            md_lines.append(f"**{DISPLAY_NAMES[k]}** [R1 {sc['rouge1']} / R2 {sc['rouge2']} / RLsum {sc['rougeLsum']}]: "
                            f"{PREDS[k][i].replace(chr(10), ' ')}\n")
    SAMPLES.append(item)
display(Markdown("\n".join(md_lines)))

pred_df = pd.DataFrame({"id": test_df["id"].astype(str), "reference": REFS, **{k: PREDS[k] for k in PREDS}})
pred_df.to_csv(os.path.join(RESULTS_DIR, "test_predictions.csv"), index=False)
print("saved results/test_predictions.csv", pred_df.shape)

In [ ]:
TEST_CASES = {
    '01_news_electric_buses': """The municipal corporation of a mid-sized hill city has approved a plan to replace its entire diesel bus fleet with 120 electric buses by the end of 2028, officials said on Monday. The first batch of 30 buses is expected to arrive in March and will run on the four busiest routes, which together carry about 65,000 passengers a day.

The project will cost an estimated 210 crore rupees, with roughly half the money coming from a national clean-transport grant and the rest from a low-interest loan. Two new depots with fast chargers will be built on the northern and southern edges of the city, each able to charge 40 buses overnight.

Transport officials said the switch would cut the corporation's fuel bill by nearly 40 per cent and remove around 9,000 tonnes of carbon dioxide emissions every year. Ticket prices will stay the same for at least two years, they added.

Not everyone is convinced. A local commuters' association welcomed the cleaner buses but warned that steep hill roads could drain batteries faster than expected, especially during the monsoon, when air-conditioning use rises. The association asked the corporation to publish range data from a three-month trial before placing the full order.

The corporation said a trial with two leased buses had already covered 18,000 kilometres without a single breakdown, and that the tender requires manufacturers to guarantee a range of at least 220 kilometres on a single charge under full load.""",
    '02_news_flood_warning_system': """Villages along a flood-prone river basin in eastern India will receive flood warnings by SMS and voice call up to 36 hours in advance under a new early-warning system launched by the state disaster management authority on Thursday.

The system combines rainfall readings from 140 automatic weather stations with water-level sensors installed on 26 bridges and embankments. A hydrological model updates forecasts every hour and sends alerts in Hindi, English and two tribal languages to registered mobile numbers in the affected blocks. Villagers without phones will be warned through loudspeakers mounted on panchayat buildings.

Officials said the pilot, which ran in three districts during last year's monsoon, sent 41 alerts and correctly predicted 37 of the 39 floods that occurred, with an average lead time of 22 hours. Two alerts turned out to be false alarms. During the pilot, the number of cattle lost to flooding in the three districts fell by more than half compared with the previous five-year average.

The authority plans to extend the network to all 24 districts of the state within two years at a cost of about 85 crore rupees. It is also training 3,000 volunteers, most of them women from self-help groups, to guide evacuations and maintain the sensors.

Experts welcomed the move but cautioned that warnings are only useful if people trust them. A hydrologist who reviewed the pilot said the authority should publish its accuracy figures every season and explain every false alarm publicly, so that villagers do not start ignoring the messages.""",
    '03_research_abstract': """Large language models are increasingly deployed to summarize scientific articles, yet their summaries frequently contain statements that are not supported by the source document. In this work we study how the length of the input context affects the factual consistency of abstractive summaries of biomedical papers. We construct a benchmark of 2,400 open-access articles paired with expert-written lay summaries and annotate 9,600 model-generated sentences for unsupported claims. We compare three strategies for handling documents that exceed the model's context window: truncation to the first 1,024 tokens, retrieval of the most relevant sections, and hierarchical summarization in which each section is summarized independently before a final summary is produced. Hierarchical summarization improves ROUGE-L by 2.1 points over truncation and reduces the rate of unsupported sentences from 18.4 per cent to 11.7 per cent, whereas retrieval gives smaller gains on both measures. Most remaining errors involve numerical values and the direction of reported effects, which suggests that numeric grounding is a promising target for future work. We release the benchmark, the annotations and our evaluation code to support research on faithful summarization of long scientific documents.""",
    '04_long_paper_token_pruning': """Adaptive Token Pruning for Efficient Transformer Inference on Edge Devices

Abstract. Transformer models achieve strong accuracy on language and vision tasks, but their inference cost grows quadratically with the number of input tokens, which makes deployment on phones and embedded boards difficult. We propose an adaptive token pruning method that removes uninformative tokens layer by layer, using a lightweight scoring head trained jointly with the model. On three text classification benchmarks and one image classification benchmark, the method reduces inference latency by 38 to 52 per cent while losing less than one percentage point of accuracy.

1. Introduction. Edge devices such as smartphones, drones and industrial sensors increasingly run neural networks locally in order to protect privacy, reduce network traffic and respond in real time. Transformers are attractive for these settings because a single architecture can handle text, images and audio. However, self-attention compares every token with every other token, so the computation and memory required grow with the square of the sequence length. Many tokens contribute little to the final prediction. In a sentence classification task, for example, punctuation, articles and repeated words can often be ignored after the first few layers without changing the output. Existing pruning methods usually remove a fixed fraction of tokens at predefined layers. Such fixed schedules are simple, but they waste computation on easy inputs and prune too aggressively on difficult ones. Our goal is a pruning policy that adapts to each input and to the latency budget of the target device.

2. Method. We attach a small scoring head, consisting of a single linear layer followed by a sigmoid, to selected transformer layers. The head assigns each token a keep probability based on its hidden state. During training, tokens are sampled with a straight-through estimator so that the scoring head receives gradients from the task loss. We add a budget loss that penalizes the expected number of floating-point operations whenever it exceeds a target value chosen by the user. At inference time, tokens whose keep probability falls below a threshold are removed, and the remaining tokens are packed into a shorter sequence, which reduces the cost of all following layers. The classification token is never pruned. Because the threshold can be changed after training, a single trained model can serve devices with different latency budgets.

3. Experimental setup. We evaluate the method on three text benchmarks covering sentiment analysis, topic classification and natural language inference, and on one image benchmark with one thousand classes. For text we fine-tune a twelve-layer encoder with 110 million parameters; for images we use a vision transformer of similar size that splits each image into 196 patches. Latency is measured on a mid-range smartphone processor and on a single-board computer with an embedded graphics unit, using batch size one, which reflects typical on-device usage. We compare against the unpruned model, two fixed-schedule pruning methods and a distilled six-layer model.

4. Results. On the text benchmarks, adaptive pruning reduces average latency by 38 per cent on the smartphone and 44 per cent on the single-board computer, with an average accuracy drop of 0.6 percentage points. Fixed-schedule pruning achieves similar speed-ups only at the cost of accuracy drops between 1.8 and 3.1 points. The distilled model is faster than all pruning variants, but it loses 2.4 points of accuracy on natural language inference, the most difficult task. On the image benchmark, adaptive pruning removes on average 61 per cent of the patches by the final layer and reduces latency by 52 per cent with a top-1 accuracy drop of 0.9 points. Visualizations show that the retained patches concentrate on the object, while background regions such as sky and walls are removed early.

5. Analysis. The number of retained tokens varies widely between inputs. Short, unambiguous reviews keep fewer than a quarter of their tokens after the sixth layer, whereas long reviews with negation or sarcasm keep more than half. This behaviour explains why adaptive pruning preserves accuracy better than fixed schedules. An ablation study shows that the budget loss is essential: without it, the scoring head learns to keep almost every token, and the speed-up disappears. Scoring heads placed only in the middle layers give the best trade-off, because early layers have not yet built useful representations and late layers leave little computation to save.

6. Limitations. Our experiments use encoder-only models and classification tasks. Generative tasks such as summarization and translation require the decoder to attend to pruned positions, which our method does not yet support. In addition, latency gains depend on the efficiency of the sequence-packing operation, which is well optimized on the processors we tested but may be slower on other hardware.

7. Conclusion. Adaptive token pruning offers a simple way to make transformer inference cheaper on edge devices while keeping accuracy close to that of the full model. Because the pruning threshold can be adjusted after training, one model can be deployed across devices with different performance limits. Future work will extend the approach to encoder-decoder models and combine it with weight quantization.""",
}

for _name, _text in TEST_CASES.items():
    with open(os.path.join(WORK_DIR, "test_cases", f"{_name}.txt"), "w", encoding="utf-8") as f:
        f.write(_text + "\n")

CUSTOM, md_lines = [], []
_budget = summarizer.max_source_len - summarizer.count_tokens(summarizer.prefix) - 8
for name, text in TEST_CASES.items():
    n_tok = summarizer.count_tokens(summarizer.prepare(text))
    t0 = time.time()
    if n_tok > _budget:
        out = summarizer.summarize_long(text)
        abstractive, mode = out["summary"], f"long-document map-reduce: {out['chunks']} chunks, {out['rounds']} round(s)"
    else:
        out, abstractive, mode = None, summarizer.summarize(text), "single pass, adaptive length"
    secs = time.time() - t0
    nov = novel_ngram_ratio([text], [abstractive], ns=(1, 2))
    item = dict(name=name, words=count_words(text), tokens=n_tok, mode=mode, seconds=round(secs, 2),
                abstractive=abstractive, textrank=textrank(text, 3), lead3=lead_k(text, 3),
                novel_unigrams_pct=nov["1"], novel_bigrams_pct=nov["2"])
    if out:
        item["partial_summaries"] = out["partial_summaries"]
    CUSTOM.append(item)
    md_lines.append(f"### {name}\n*{item['words']} words / {n_tok} tokens — {mode} — {secs:.1f}s — "
                    f"novel bigrams {nov['2']}%*\n\n**Abstractive ({MODEL_TAG} fine-tuned):** {abstractive}\n\n"
                    f"**TextRank:** {item['textrank'].replace(chr(10), ' ')}\n\n**LEAD-3:** {item['lead3'].replace(chr(10), ' ')}\n")
    if out:
        md_lines.append("<details><summary>partial (chunk) summaries</summary>\n\n" +
                        "\n".join(f"{j}. {p}" for j, p in enumerate(out["partial_summaries"][0], 1)) + "\n</details>\n")
CUSTOM_MD = "\n".join(md_lines)
with open(os.path.join(RESULTS_DIR, "custom_cases.md"), "w", encoding="utf-8") as f:
    f.write(f"# Custom test cases — {MODEL_TAG} fine-tuned on CNN/DailyMail\n\n" + CUSTOM_MD)
display(Markdown(CUSTOM_MD))

## 8. Save metrics, model card and outputs
`results/metrics.json` contains everything (configuration, environment, data statistics, training log, all scores, significance tests, examples, custom cases). It is the input for the project report.

In [ ]:
def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_jsonable(o.tolist())
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    return o


METRICS = dict(
    project=dict(title="Text Summarization with Transformers (CNN/DailyMail)", author="Kumar Sidharth",
                 repo="https://github.com/ksidharth8/text-summarization-transformers"),
    run_mode=RUN_MODE, timestamp=datetime.datetime.now().isoformat(timespec="seconds"), model_name=CFG["model_name"],
    config=CFG, environment=ENV,
    data=dict(files=DATA_FILES, split_sizes=SPLIT_SIZES, dropped=DROPPED,
              used=dict(train=TRAIN_INFO["train_examples"], validation=TRAIN_INFO["val_examples"], test=N_TEST),
              stats=DATA_STATS, cleaning_example=CLEAN_EXAMPLE),
    training=TRAIN_INFO, generation=GEN_INFO, results=RESULTS, significance=SIGNIFICANCE,
    zero_shot_subset=dict(n=len(ZS_IDX), scores=ZS_SUBSET), rouge1_by_article_length=BY_LENGTH,
    timing_seconds=TIMING, samples=SAMPLES, custom_cases=CUSTOM, figures=FIGURES,
    total_runtime_hours=round((time.time() - T_NOTEBOOK) / 3600, 3))
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(to_jsonable(METRICS), f, indent=2, ensure_ascii=False)

rows = ["| System | ROUGE-1 | ROUGE-2 | ROUGE-L | ROUGE-Lsum | avg. words |", "|---|---|---|---|---|---|"]
for k in SYSTEMS:
    r = RESULTS[k]
    name = DISPLAY_NAMES[k] + (f" (n={r['n']})" if r["n"] != N_TEST else "")
    rows.append(f"| {name} | {r['rouge1']:.2f} | {r['rouge2']:.2f} | {r['rougeL']:.2f} | {r['rougeLsum']:.2f} | {r['avg_words']} |")
RESULTS_TABLE = "\n".join(rows)
with open(os.path.join(RESULTS_DIR, "results_table.md"), "w") as f:
    f.write(RESULTS_TABLE + "\n")

ft = RESULTS["finetuned"]
card = f"""---
language: en
license: apache-2.0
base_model: {CFG['model_name']}
tags: [summarization, bart, cnn_dailymail]
datasets: [cnn_dailymail]
---
# {MODEL_TAG} fine-tuned on CNN/DailyMail

Abstractive news summarizer from the project
[text-summarization-transformers](https://github.com/ksidharth8/text-summarization-transformers) (Kumar Sidharth, BIT Mesra).

**Test set ({N_TEST:,} articles):** ROUGE-1 {ft['rouge1']:.2f} · ROUGE-2 {ft['rouge2']:.2f} · ROUGE-L {ft['rougeL']:.2f} · ROUGE-Lsum {ft['rougeLsum']:.2f}

Training: {TRAIN_INFO['steps_completed']:,} steps × batch {EFFECTIVE_BATCH} ({TRAIN_INFO['epochs_completed']} epochs, {TRAIN_INFO['train_runtime_hours']} h on {ENV['gpu']}),
input ≤ {CFG['max_source_len']} tokens, AdamW lr {CFG['lr']}, label smoothing {CFG['label_smoothing']}, fp16. Run mode: `{RUN_MODE}`.

```python
from transformers import pipeline
summarize = pipeline("summarization", model="PATH_OR_HUB_ID")
print(summarize(article, num_beams=4, max_new_tokens=142, min_new_tokens=56, length_penalty=2.0,
                no_repeat_ngram_size=3, truncation=True)[0]["summary_text"])
```
or, with long-document support, `src.summarizer.Summarizer("PATH")` from the repository.
"""
with open(os.path.join(MODEL_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(card)
with zipfile.ZipFile(MODEL_ZIP, "a") as zf:
    zf.write(os.path.join(MODEL_DIR, "README.md"), os.path.join(os.path.basename(MODEL_DIR), "README.md"))
display(Markdown(RESULTS_TABLE))
print("saved results/metrics.json, results/results_table.md and the model card")

In [ ]:
# Optional: publish the model on the Hugging Face Hub (Kaggle: Add-ons -> Secrets -> HF_TOKEN, then push_to_hub=True)
if CFG["push_to_hub"] and RUN_MODE == "full":
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import HfApi
        token = UserSecretsClient().get_secret("HF_TOKEN")
        api = HfApi(token=token)
        api.create_repo(CFG["hub_repo"], exist_ok=True)
        api.upload_folder(folder_path=MODEL_DIR, repo_id=CFG["hub_repo"], commit_message="fine-tuned on CNN/DailyMail")
        print("pushed to https://huggingface.co/" + CFG["hub_repo"])
    except Exception as e:
        print("push to hub failed:", repr(e)[:300])
else:
    print("push_to_hub disabled")

In [ ]:
shutil.rmtree(CKPT_DIR, ignore_errors=True)
print(f"total runtime: {(time.time() - T_NOTEBOOK) / 3600:.2f} h\n\nOutputs in {WORK_DIR}:")
for root, dirs, files in os.walk(WORK_DIR):
    dirs[:] = sorted(d for d in dirs if not d.startswith(".") and d != "__pycache__")
    depth = root[len(WORK_DIR):].count(os.sep)
    if depth > 2:
        continue
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, WORK_DIR):<60} {os.path.getsize(p) / 1e6:9.2f} MB")
if RUN_MODE == "smoke":
    print("\n" + "!" * 90 + "\nSMOKE RUN FINISHED - pipeline works, but the scores are meaningless (tiny training run).\n"
          "Now set RUN_MODE = \"full\" and use 'Save Version -> Save & Run All (Commit)'.\n" + "!" * 90)